In [5]:
%pip install nbformat>=4.2.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
from scipy.stats import skew
import os

# Create mandatory project folders for GitHub/Pipeline compliance
for folder in ['data', 'outputs']:
    if not os.path.exists(folder):
        os.makedirs(folder)
        print(f"Created directory: {folder}")

In [7]:
# Load the verified dataset
file_path = 'HVAC_NE_EC_19-21.csv'

try:
    df = pd.read_csv(file_path)
    # Save the original to the data folder per architecture requirements
    df.to_csv('data/dataset_original.csv', index=False)
    print(f"[SUCCESS] AHU Telemetry Ingested: {len(df)} rows found.")
    display(df.head()) # Shows the first 5 rows in your notebook
except Exception as e:
    print(f"[ERROR] Ingestion failed: {e}")

    

[SUCCESS] AHU Telemetry Ingested: 33888 rows found.


,Timestamp,T_Supply,T_Return,SP_Return,T_Saturation,T_Outdoor,RH_Supply,RH_Return,RH_Outdoor,Energy,Power
0,2019-10-15 00:00:00+02:00,19.859999,20.469999,18.5,19.02,20.299999,71.110001,58.919998,79.5,0.0,0.0
1,2019-10-15 00:15:00+02:00,19.855000,20.430000,18.5,19.02,20.299999,71.320000,59.000000,82.0,0.0,0.0
2,2019-10-15 00:30:00+02:00,19.850000,20.410000,18.5,19.02,20.299999,71.470001,59.109997,79.5,0.0,0.0
3,2019-10-15 00:45:00+02:00,19.840000,20.379999,18.5,19.08,20.299999,71.439995,59.309998,77.0,0.0,0.0
4,2019-10-15 01:00:00+02:00,19.830000,20.350000,18.5,19.08,20.299999,71.580002,59.559998,79.5,0.0,0.0


In [8]:
# Data Cleaning
cleaned_df = df.dropna().drop_duplicates()

# UNIQUE FILTER LOGIC: Isolating high-load periods (> 5.0 kW)
# In Duct Leakage analysis, high-load cycles reveal pressure-drop anomalies.
if 'Power' in cleaned_df.columns:
    cleaned_df = cleaned_df[cleaned_df['Power'] > 5.0]

# Save cleaned data to the mandatory data folder
cleaned_df.to_csv('data/dataset_cleaned.csv', index=False)
print(f"[SUCCESS] Unique Filter applied. Sample size: {len(cleaned_df)} rows.")

[SUCCESS] Unique Filter applied. Sample size: 7439 rows.


In [9]:
# Convert to NumPy for high-performance engineering calculations
power_vals = cleaned_df['Power'].to_numpy()

findings = {
    "Mean_Load (kW)": np.mean(power_vals),
    "Load_Variance": np.var(power_vals),
    "Load_Skewness": skew(power_vals),
    "Peak_Observation": np.max(power_vals)
}

print("--- ENGINEERING METRICS FOR HVA-04 ---")
for key, value in findings.items():
    print(f"{key}: {value:.4f}")

--- ENGINEERING METRICS FOR HVA-04 ---
Mean_Load (kW): 5.1193
Load_Variance: 0.0057
Load_Skewness: 0.1084
Peak_Observation: 5.3160


In [10]:
# 1. Static Scatter Plot: Pressure vs. Power (The "Leakage Signature")
fig_static = px.scatter(cleaned_df, x='SP_Return', y='Power', 
                         title="HVA-04: Duct Pressure vs. Power Correlation")
fig_static.write_image("outputs/static_scatter.png")

# 2. Animated Behavior over Time
# We use a slice (first 500 rows) to keep the animation smooth in the browser
fig_anim = px.scatter(cleaned_df.head(500), x='SP_Return', y='Power',
                      animation_frame='Timestamp', 
                      title="Dynamic Pressure-Power Fluctuations")
fig_anim.write_html("outputs/animated_leakage.html")

print("[SUCCESS] Static PNG and Animated HTML saved to /outputs folder.")
fig_static.show() # Display the plot directly in your notebook

[SUCCESS] Static PNG and Animated HTML saved to /outputs folder.
